# 02 — Training Analysis & Debugging (Day 9)

Understand *why* the dual-tower model learns, before Gate 1 (Day 10). Five diagnostic panels + an `embed_dim` ablation + a domain-collapse check.

The trainer records per-encoder gradient norms and accepts an `on_epoch_end` hook, which we use to snapshot validation embeddings each epoch without duplicating the training loop. Checkpoints are written to a scratch path so Day 8's `models/best_model.pt` (the Gate 1 model) is left untouched. `mlflow` is disabled here to keep the analysis run out of the `pctrans-v1` experiment.

Observed values in the markdown cells are from the executed run of this notebook (seed 42; CCLE val N=38, TCGA val N=339). Numbers are also serialised to `reports/day9_history.json`.

In [ ]:
import json
import pickle
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.manifold import TSNE

from pctrans.data.dataset import IDX_TO_LINEAGE, CCLEDataset, TCGADataset
from pctrans.data.preprocessor import DataSplitter
from pctrans.data.sampler import StratifiedContrastiveBatchSampler
from pctrans.models.dual_tower import DualTowerModel
from pctrans.models.encoders import CCLEEncoder, TCGAEncoder
from pctrans.models.losses import SupConInfoNCELoss
from pctrans.training.trainer import ContrastiveTrainer

DATA_DIR = Path('../data/processed')
REPORTS = Path('../reports')
SCRATCH = Path(tempfile.mkdtemp(prefix='pctrans_day9_'))  # throwaway checkpoints
LINEAGE_COLOR = {'LUAD': '#1f77b4', 'BRCA': '#e377c2', 'SKCM': '#8c564b'}


def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)


train_cfg = yaml.safe_load((Path('../configs/training.yaml')).read_text())
model_cfg = yaml.safe_load((Path('../configs/model.yaml')).read_text())
ccle_df = pd.read_parquet(DATA_DIR / 'ccle_2k.parquet')
tcga_df = pd.read_parquet(DATA_DIR / 'tcga_2k.parquet')
splits = json.loads((DATA_DIR / 'splits.json').read_text())
with open(DATA_DIR / 'scalers.pkl', 'rb') as f:
    scalers = pickle.load(f)
splitter = DataSplitter()


def scaled(df, ids):
    return splitter.apply_scalers(df.loc[ids], scalers)


ccle_train = CCLEDataset(scaled(ccle_df, splits['ccle']['train']))
ccle_val = CCLEDataset(scaled(ccle_df, splits['ccle']['val']))
tcga_train = TCGADataset(scaled(tcga_df, splits['tcga']['train']))
tcga_val = TCGADataset(scaled(tcga_df, splits['tcga']['val']))
print(f'val: CCLE {len(ccle_val)} + TCGA {len(tcga_val)}')


def build(embed_dim=None):
    hidden = tuple(model_cfg.get('hidden_dims', [1024, 512, 256, 128]))
    kwargs = dict(
        input_dim=model_cfg.get('input_dim', 2000),
        hidden_dims=hidden,
        embed_dim=embed_dim or model_cfg.get('embed_dim', 64),
        dropout=model_cfg.get('dropout_high', 0.3),
        dropout_low=model_cfg.get('dropout_low', 0.2),
    )
    return (
        DualTowerModel(CCLEEncoder(**kwargs), TCGAEncoder(**kwargs)),
        SupConInfoNCELoss(init_tau=model_cfg.get('init_tau', 0.07)),
    )

## Main training run

Full config (30 epochs, early-stop patience 5). The `snapshot` hook stores L2-normalised validation embeddings for both towers each epoch, so Panel 5 can compare epoch 1 against the best epoch.

In [ ]:
seed_everything(42)
model, loss_fn = build()
cfg = dict(train_cfg)
cfg['checkpoint_path'] = str(SCRATCH / 'day9_best.pt')
sampler = StratifiedContrastiveBatchSampler(ccle_train, tcga_train, batch_size=cfg['batch_size'])
trainer = ContrastiveTrainer(model, loss_fn, sampler, ccle_val, tcga_val, cfg, mlflow_run_name=None)

val_embeddings = {}


def snapshot(epoch, m, record):
    m.eval()
    with torch.no_grad():
        zc = m.encode_ccle(ccle_val.features).cpu().numpy()
        zt = m.encode_tcga(tcga_val.features).cpu().numpy()
    val_embeddings[epoch] = (zc, zt)


result = trainer.train(on_epoch_end=snapshot)
history = result['history']
best_epoch = result['best_epoch']
epochs = [r['epoch'] + 1 for r in history]
y_ccle_val, y_tcga_val = ccle_val.labels, tcga_val.labels
print(f"ran {len(history)} epochs; best epoch {best_epoch + 1}; "
      f"best val kNN@5 {result['best_val_knn_accuracy']:.4f}")

### Panel 1 — train loss vs. val kNN@5

Dual axis: train loss (log scale) falling while val kNN@5 rises. The dashed line is the Gate 1 threshold (0.70); the dotted vertical is the best epoch.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(epochs, [r['train_loss'] for r in history], 'o-', color='#d62728', label='train loss')
ax1.set_yscale('log')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss (log)', color='#d62728')
ax1.tick_params(axis='y', labelcolor='#d62728')
ax2 = ax1.twinx()
ax2.plot(epochs, [r['val_knn_accuracy'] for r in history], 's-', color='#2ca02c', label='val kNN@5')
ax2.axhline(0.70, ls='--', color='gray', lw=1)
ax2.set_ylabel('val kNN@5', color='#2ca02c'); ax2.set_ylim(0, 1)
ax2.tick_params(axis='y', labelcolor='#2ca02c')
ax1.axvline(best_epoch + 1, ls=':', color='black', lw=1)
ax1.set_title('Panel 1 - train loss vs. val kNN@5')
fig.tight_layout(); fig.savefig(REPORTS / 'day9_panel1_loss_knn.png', dpi=120); plt.show()

**Observed:** val kNN@5 is already 0.842 after epoch 1 and jumps to **0.9737 at epoch 2 (the best epoch)**. Train loss falls monotonically 7.53 → 4.19; val loss bottoms at epoch 2 (7.76) then drifts up — classic mild overfitting on a tiny (N=183 CCLE) train set, which is exactly why early stopping checkpoints epoch 2. Training stops at epoch 7 (patience 5).

### Panel 2 — learned temperature evolution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, [r['temperature'] for r in history], 'o-', color='#9467bd')
ax.set_xlabel('epoch'); ax.set_ylabel('temperature tau')
ax.set_title('Panel 2 - learned temperature evolution'); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(REPORTS / 'day9_panel2_temperature.png', dpi=120); plt.show()

**Observed:** tau drifts **0.0702 -> 0.0734**, essentially pinned at its 0.07 init. It does *not* fall into the 0.02-0.05 range that literature associates with well-separated CLIP-style embeddings. Interpretation: the lineage signal is strong enough that the model hits >0.95 kNN without needing to sharpen the temperature; over only ~66 batches/epoch x 7 epochs the learnable `log(1/tau)` barely moves. Crucially tau does **not** collapse toward 0 (which would signal representation collapse) — it stays stable, slightly rising.

### Panel 3 — per-lineage val kNN@5

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for lin in ('LUAD', 'BRCA', 'SKCM'):
    ax.plot(epochs, [r['per_lineage'].get(lin, np.nan) for r in history], 'o-',
            color=LINEAGE_COLOR[lin], label=lin)
ax.axhline(0.40, ls='--', color='gray', lw=1, label='0.40 floor')
ax.set_xlabel('epoch'); ax.set_ylabel('per-lineage val kNN@5'); ax.set_ylim(0, 1.05)
ax.set_title('Panel 3 - per-lineage val kNN@5'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(REPORTS / 'day9_panel3_per_lineage_knn.png', dpi=120); plt.show()

**Observed (best epoch 2):** LUAD 1.00, BRCA 1.00, SKCM 0.9375. All lineages sit far above the 0.40 diagnostic floor, so the sampler's class balance is fine.

- **SKCM is the hardest to translate**: it caps at 0.9375 (15/16) at every epoch — one melanoma   cell line never retrieves SKCM patients. This matches the hypothesis that culture-driven   melanocyte markers dominate over the tumour-microenvironment signal present in patients.
- **BRCA is the most volatile**: 1.00 at epoch 2, dropping to 0.70 at epochs 3-4 before   recovering to 0.90. On a 10-sample val slice each step is one cell line, so this is small-N   jitter rather than a systematic failure.
- **LUAD is perfect** from epoch 2 onward.

### Panel 4 — gradient norm per encoder tower

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, [r['grad_norm_ccle'] for r in history], 'o-', color='#1f77b4', label='CCLEEncoder')
ax.plot(epochs, [r['grad_norm_tcga'] for r in history], 's-', color='#ff7f0e', label='TCGAEncoder')
ax.set_xlabel('epoch'); ax.set_ylabel('mean per-batch grad L2 norm (pre-clip)')
ax.set_title('Panel 4 - gradient norm per encoder tower'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(REPORTS / 'day9_panel4_grad_norms.png', dpi=120); plt.show()

**Observed:** the two towers track each other closely for the whole run — no 10x asymmetry, so neither tower is starved.

| epoch | CCLE | TCGA |
|---|---|---|
| 1 | 41.7 | 42.0 |
| 2 | 25.8 | 21.4 |
| 3 | 10.7 |  7.1 |
| 4 |  2.9 |  1.8 |
| 5 |  0.55 | 0.74 |
| 6 |  0.27 | 0.47 |
| 7 |  0.18 | 0.44 |

During the high-signal epochs (1-4) the **CCLE tower's gradient is equal-or-slightly-larger** — it has more to learn because CCLE starts poorly aligned (see Panel 5). After convergence (epochs 5-7) both gradients decay below 1.0 and TCGA is marginally higher; the ~2.4x final-epoch ratio is on essentially-zero gradients and should not be read as an asymmetry. Net: the towers are **balanced**.

### Panel 5 — t-SNE of validation embeddings, epoch 1 vs. best epoch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, ep, title in [(axes[0], 0, 'epoch 1'), (axes[1], best_epoch, f'best epoch ({best_epoch + 1})')]:
    zc, zt = val_embeddings[ep]
    coords = TSNE(n_components=2, perplexity=30, init='pca', random_state=42).fit_transform(np.vstack([zc, zt]))
    cc, ct = coords[:len(zc)], coords[len(zc):]
    for idx, lin in IDX_TO_LINEAGE.items():
        ax.scatter(ct[y_tcga_val == idx, 0], ct[y_tcga_val == idx, 1], c=LINEAGE_COLOR[lin],
                   marker='o', s=18, alpha=0.5, label=f'{lin} TCGA')
        ax.scatter(cc[y_ccle_val == idx, 0], cc[y_ccle_val == idx, 1], c=LINEAGE_COLOR[lin],
                   marker='X', s=90, edgecolors='black', linewidths=0.5, label=f'{lin} CCLE')
    ax.set_title(f'val embeddings, {title}'); ax.set_xticks([]); ax.set_yticks([])
axes[1].legend(fontsize=7, markerscale=0.8, ncol=2)
fig.tight_layout(); fig.savefig(REPORTS / 'day9_panel5_tsne.png', dpi=120); plt.show()

**Observed:** at **epoch 1** the TCGA patients (circles) already separate into three lineage clouds, but the CCLE cell lines (X) are clumped together in the centre — they cluster by *domain*, not yet by lineage. By the **best epoch** the CCLE markers have migrated toward their lineage's TCGA cloud (SKCM crosses sit beside the SKCM patient cluster, LUAD near LUAD). This is the visual counterpart to Panel 4: the CCLE tower carries the larger early gradient precisely because it is the one that has to move.

## Domain-collapse check

Alignment should pull *same-lineage* samples together across domains without erasing the domain distinction entirely. We compute the centroid of all CCLE val embeddings and of all TCGA val embeddings (64-dim, best epoch) and take their cosine similarity. Target: **< 0.5** (domains still distinguishable).

In [ ]:
zc_best, zt_best = val_embeddings[best_epoch]
cen_ccle, cen_tcga = zc_best.mean(0), zt_best.mean(0)
cos = float(cen_ccle @ cen_tcga / (np.linalg.norm(cen_ccle) * np.linalg.norm(cen_tcga)))
print(f'CCLE vs TCGA centroid cosine sim = {cos:.4f}  ->  {"PASS" if cos < 0.5 else "FAIL"} (<0.5)')

**Observed:** cosine similarity **0.251 < 0.5 -> PASS**. The two domain centroids remain well separated, so the encoders align lineages without collapsing CCLE and TCGA onto a single point. No domain collapse.

## `embed_dim` ablation (3 epochs each)

Train embed_dim in {32, 64, 128} for 3 epochs each (early stopping disabled) and compare val kNN@5. Rule from the plan: prefer the simpler model, but keep 64 as the default unless the evidence clearly favours a change.

In [ ]:
ablation = {}
for ed in (32, 64, 128):
    seed_everything(42)
    m, lf = build(embed_dim=ed)
    acfg = dict(train_cfg); acfg['checkpoint_path'] = str(SCRATCH / f'day9_ablate_{ed}.pt')
    acfg['early_stop_patience'] = 0
    smp = StratifiedContrastiveBatchSampler(ccle_train, tcga_train, batch_size=acfg['batch_size'])
    tr = ContrastiveTrainer(m, lf, smp, ccle_val, tcga_val, acfg, mlflow_run_name=None)
    res = tr.train(n_epochs=3)
    ablation[ed] = (res['best_val_knn_accuracy'], res['history'][-1]['val_knn_accuracy'])
for ed, (best, final) in ablation.items():
    print(f'embed_dim={ed:>3}: best {best:.4f}  final {final:.4f}')

**Observed (best / final val kNN@5 over 3 epochs):**

| embed_dim | best | final |
|---|---|---|
| 32 | 0.9474 | 0.9474 |
| **64** | **0.9737** | **0.9737** |
| 128 | 0.9474 | 0.8947 |

All three clear the 0.70 Gate 1 threshold comfortably. On a 38-sample CCLE val set one cell line is worth 2.6%, so 0.9474 vs 0.9737 is a **one-sample** difference — 32 and 64 are statistically indistinguishable here. 128 is the only one that *drops* between its best and final epoch (0.9474 -> 0.8947), a hint of over-parameterisation at this sample size.

**Decision: keep `embed_dim = 64`.** It scored highest and most stably, it is the dimension of the Day 8 `best_model.pt` that Gate 1 will evaluate, and the whole downstream stack is already 64-dim. A stricter reading of the 'simpler-model' rule would allow 32 (it also clears 70%), but the val set cannot separate the two, so there is no evidence-based reason to retrain at 32.